# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://mlcroissant.readthedocs.io/) library. We will:

1. Load metadata and review dataset structure.
2. List and explore available record sets and fields (referenced by their `@id`).
3. Extract records and fields for exploration.
4. Conduct basic exploratory data analysis (EDA) and visualize selected aspects.
5. Summarize key insights.

### Dataset Source
The dataset is provided in Croissant schema format at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading

Load the dataset's metadata from the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata fields
print(f"Dataset Name: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Authors: {dataset.metadata.author}\n")
print(f"Record Sets: {dataset.metadata.record_set}\n")

## 2. Data Overview

Let's list all record sets, and for each, their fields and columns. Every entity is referenced by its `@id`. These `@id`s will be used for all further extraction and manipulation.

In [ ]:
print("Available Record Sets (@id):\n")
for record_set in dataset.metadata.record_set:
    print(f"- {record_set['@id']}  (name: {record_set.get('name', 'N/A')})")
    # List fields
    if 'field' in record_set:
        print("  Fields:")
        for field in record_set['field']:
            print(f"    - {field['@id']}  (name: {field.get('name', 'N/A')})")
            # List columns in this field
            if 'column' in field:
                print("      Columns:")
                for column in field['column']:
                    print(f"        - {column['@id']}")
    print("")

## 3. Data Extraction

Let's extract data from one or more record sets into DataFrame(s). For this demonstration, **all record sets** and their main fields will be loaded. 

**Tip:** Make sure to use the proper `@id` values from the overview above. If your dataset is large, consider extracting only one record set at a time.

In [ ]:
# Select record set IDs
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_set]

dataframes = {}
for rs_id in record_set_ids:
    print(f"\nLoading record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Extracted DataFrame columns for {rs_id}: {df.columns.tolist()}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)

We will process and analyze some of the loaded data. For this, pick a numeric field from your extracted DataFrame (referenced always by its field `@id`) and demonstrate filtering, normalization, and grouping. 

Replace `<RECORD_SET_ID>`, `<NUMERIC_FIELD_ID>`, and `<GROUP_FIELD_ID>` with those from your specific dataset.

Below is an example for the main record set, with placeholder IDs replaced by actual values from the dataset if available.

In [ ]:
# Example usage with the primary record set and fields
# Update these IDs as per overview above for your dataset

# First, pick the first available record set and scan numeric columns
main_record_set_id = record_set_ids[0] if record_set_ids else None

if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    # List possible numeric fields (float or int)
    num_fields = df.select_dtypes(include=['float', 'int']).columns.tolist()
    print(f"Numeric fields in {main_record_set_id}: {num_fields}")
    if num_fields:
        numeric_field = num_fields[0]
        threshold = df[numeric_field].mean() if df[numeric_field].dtype.kind in 'fi' else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
        display(filtered_df.head())
        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try grouping by another field if one exists (string type)
        str_fields = df.select_dtypes(include=['object']).columns.tolist()
        group_field = str_fields[0] if str_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
    else:
        print(f"No numeric columns found in {main_record_set_id}")
else:
    print("No record sets found in the dataset.")

## 5. Visualization

Plot distributions for numeric fields and relationships. You can use `matplotlib`, `seaborn`, or `pandas` built-in plotting. Example below uses the primary numeric field from the previous section.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and num_fields:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    if group_field:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Not enough numeric fields for visualization.")

## 6. Conclusion

- We demonstrated how to use `mlcroissant` to load, explore, and process data packages described with the Croissant schema.
- Record set and field exploration relied on their unique `@id` for robust extraction.
- You saw EDA basics: filtering by numeric thresholds, normalizing columns, and examining group-wise statistics.
- Visual insight via histograms or boxplots can help understand distributions across fields and groups.

**Next steps:**
- Review the Croissant documentation for more advanced uses.
- Extend this notebook to further process, clean, combine, or analyze the dataset.

For up-to-date schema metadata and user guides, visit: https://mlcommons.org/croissant/  

Happy data exploring!